In [40]:
import pandas as pd
import numpy as np
from datetime import datetime
import os


In [41]:
file_raw_trading = r'C:\Dự án tốt nghiêp\crwaling_data\raw_trading_data.csv'
file_raw_candel5m = r'C:\Dự án tốt nghiêp\crwaling_data\raw_candel5m_data.csv'
file_raw_ticker = r'C:\Dự án tốt nghiêp\crwaling_data\raw_ticker_data.csv'


# Clean, normalize and enrich data

## Trading Data

In [42]:
def clean_trading_data(df):
    
    
    # 1. Đổi tên cột
    df = df.rename(columns={
        'ts': 'timestamp_api',
        'price': 'price',
        'size': 'volume',
        'side': 'side',
        'tradeId': 'trade_id',
        'instId': 'symbol',
        'ts_recv': 'timestamp_received'
    })
    
    # 2. Chuyển đổi timestamp
    df['timestamp_api'] = pd.to_datetime(df['timestamp_api'], unit='ms')
    df['timestamp_received'] = pd.to_datetime(df['timestamp_received'])
    
    
    # 3. Chuyển đổi kiểu dữ liệu số
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    df['volume'] = pd.to_numeric(df['volume'], errors='coerce')
    df['trade_id'] = pd.to_numeric(df['trade_id'], errors='coerce')

    # 4. Loại bỏ dữ liệu lỗi
    df = df.dropna()
    df = df[df['price'] > 0]
    df = df[df['volume'] > 0]
    df = df[df['trade_id'] > 0]
    
    # 5. Loại bỏ duplicate
    df = df.drop_duplicates(subset=['trade_id', 'symbol'])
    
    # 7. Thêm cột phân tích
    df['value_usd'] = df['price'] * df['volume']
    df['hour'] = df['timestamp_api'].dt.hour
    df['minute'] = df['timestamp_api'].dt.minute
    return df

In [43]:
df_trading = pd.read_csv(file_raw_trading)
df_trading_clean = clean_trading_data(df_trading)
df_trading_clean.to_csv('cleaned_trading_data1.csv', index=False)

## Ticker Data

In [ ]:

def clean_ticker_data(df):
    
    
    # 1. Đổi tên cột
    df = df.rename(columns={
        'instId': 'symbol',
        'lastPr': 'last_price',
        'bidPr': 'bid_price',
        'askPr': 'ask_price',
        'bidSz': 'bid_size',
        'askSz': 'ask_size',
        'open24h': 'open_24h',
        'high24h': 'high_24h',
        'low24h': 'low_24h',
        'change24h': 'change_24h_pct',
        'baseVolume': 'base_volume_24h',
        'quoteVolume': 'quote_volume_24h',
        'ts': 'timestamp_api',
        'ts_recv': 'timestamp_received'
    })
    
    
    
    # 2. Chuyển đổi timestamp
    df['timestamp_api'] = pd.to_datetime(df['timestamp_api'], unit='ms')
    df['timestamp_received'] = pd.to_datetime(df['timestamp_received'])
    
    
    
    # 3. Chuyển đổi kiểu dữ liệu số
    price_cols = ['last_price', 'bid_price', 'ask_price', 'open_24h', 'high_24h', 'low_24h']
    volume_cols = ['bid_size', 'ask_size', 'base_volume_24h', 'quote_volume_24h']
    for col in price_cols + volume_cols + ['change_24h_pct']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    

    # 4. Loại bỏ dữ liệu lỗi
    df = df.dropna(subset=['symbol', 'last_price'])
    df = df[df['last_price'] > 0]
    
    
    
    # 5. Tính toán spread
    df['spread'] = df['ask_price'] - df['bid_price']
    df['spread_pct'] = (df['spread'] / df['last_price']) * 100
    
    
    
    # # 6. Loại bỏ duplicate theo thời gian gần nhất
    # df = df.sort_values('timestamp_api').drop_duplicates(subset=['symbol'], keep='last')
    
    
    return df





In [45]:
df_ticker = pd.read_csv(file_raw_ticker)
df_ticker_clean = clean_ticker_data(df_ticker)
df_ticker_clean.to_csv('cleaned_ticker_data.csv', index=False)

## Candel5m Data

In [ ]:

def clean_candlestick_data(df):

    # 1. Đổi tên cột
    df = df.rename(columns={
        'instId': 'symbol',
        'start_time': 'timestamp_start',
        'open_price': 'open',
        'highest_price': 'high',
        'lowest_price': 'low',
        'closing_price': 'close',
        'trading_volume_coin': 'volume_base',
        'trading_volume_usd': 'volume_quote',
        'ts_recv': 'timestamp_received'
    })
    
    # 2. Chuyển đổi timestamp
    df['timestamp_start'] = pd.to_datetime(df['timestamp_start'], unit='ms')
    df['timestamp_received'] = pd.to_datetime(df['timestamp_received'])
    
    # 3. Chuyển đổi kiểu dữ liệu số
    ohlc_cols = ['open', 'high', 'low', 'close']
    volume_cols = ['volume_base', 'volume_quote']
    
    for col in ohlc_cols + volume_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 4. Loại bỏ dữ liệu lỗi
    df = df.dropna(subset=['symbol'] + ohlc_cols)
    df = df[(df['open'] > 0) & (df['high'] > 0) & (df['low'] > 0) & (df['close'] > 0)]
    
    # 5. Kiểm tra tính hợp lệ của OHLC
    df = df[df['high'] >= df[['open', 'close']].max(axis=1)]
    df = df[df['low'] <= df[['open', 'close']].min(axis=1)]
    df = df[df['high'] >= df['low']]
    
    # 6. Tính toán các chỉ số kỹ thuật
    df['price_change'] = df['close'] - df['open']
    df['price_change_pct'] = (df['price_change'] / df['open']) * 100
    df['price_range'] = df['high'] - df['low']
    df['price_range_pct'] = (df['price_range'] / df['open']) * 100
    
    # 7. Phân loại nến
    df['candle_type'] = df['price_change'].apply(lambda x: 'bullish' if x > 0 else ('bearish' if x < 0 else 'doji'))
    
    # 8. Thêm timeframe
    df['timeframe'] = df['channel'].str.replace('candle', '')
    
    # 9. Sắp xếp theo thời gian
    df = df.sort_values(['symbol', 'timestamp_start'])
    
    # 10. Loại bỏ duplicate
    df = df.drop_duplicates(subset=['symbol', 'timestamp_start', 'timeframe'])
    
    return df




In [47]:
df_candle = pd.read_csv(file_raw_candel5m)
df_candle_clean = clean_candlestick_data(df_candle)
df_candle_clean.to_csv('cleaned_candlestick_data.csv', index=False)